In [ ]:
import sys
sys.path.append("../")

In [ ]:
# imports
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
df_final = pd.read_csv('../data/large/final_df_v1.csv')

In [4]:
def process_data(df_to_use, X_numerical, response):
    label_encoder = LabelEncoder()
    scaler = StandardScaler()

    X = df_to_use[X_numerical]
    y = df_to_use[response]
    y_encoded = label_encoder.fit_transform(y)


    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded,
        test_size=0.2,
        random_state=101,
        stratify=y_encoded
    )

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    return  X_train, X_test, y_train, y_test, label_encoder, scaler

In [5]:
X_numerical = ['INCL', 'RAAN', 'ECC', 'ARG_PER', 'MEAN_MOTION', 'SMA_KM','APOGEE_KM', 'PERIGEE_KM', 'MEAN_MOTION_1ST_DER'] 
response = 'TYPE'

### Model: NNs

In [7]:
X_train, X_test, y_train, y_test, label_encoder, scaler = process_data(df_final, X_numerical, response)

early_stopping = EarlyStopping(
    monitor='val_loss', 
    patience=5,
    restore_best_weights=True,
    verbose=1
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'nn_big_v0_1.h5', 
    monitor='val_loss', 
    save_best_only=True,
    verbose=1
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.5, 
    patience=5, 
    verbose=1, 
    min_lr=1e-6
)

DROPOUT = .15

y_train_one_hot = tf.keras.utils.to_categorical(y_train, num_classes=3)

model = models.Sequential()
model.add(layers.InputLayer(input_shape=(len(X_numerical),)))
model.add(layers.Dense(500, activation='relu'))  
model.add(layers.Dropout(DROPOUT))
model.add(layers.Dense(250, activation='relu'))  
model.add(layers.Dropout(DROPOUT))
model.add(layers.Dense(50, activation='relu'))  
model.add(layers.Dropout(DROPOUT))
model.add(layers.Dense(3, activation='softmax'))  

model.compile(optimizer=Adam(learning_rate=.001), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

model.fit(
    X_train, y_train_one_hot,
    epochs=10,
    batch_size=32,
    validation_split=0.15,
    callbacks=[early_stopping, reduce_lr,checkpoint ]
)

y_test_one_hot = tf.keras.utils.to_categorical(y_test, num_classes=3)
test_loss, test_accuracy = model.evaluate(X_test, y_test_one_hot)
print(f"Test accuracy: {test_accuracy:.4f}")


Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_4 (Dense)             (None, 500)               5000      
                                                                 
 dropout_3 (Dropout)         (None, 500)               0         
                                                                 
 dense_5 (Dense)             (None, 250)               125250    
                                                                 
 dropout_4 (Dropout)         (None, 250)               0         
                                                                 
 dense_6 (Dense)             (None, 50)                12550     
                                                                 
 dropout_5 (Dropout)         (None, 50)                0         
                                                                 
 dense_7 (Dense)             (None, 3)                

/shared/courseSharedFolders/142601outer/142601/cs109b/lib/python3.11/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


21465/21485 [============================>.] - ETA: 0s - loss: 0.3366 - accuracy: 0.8616
Epoch 2: val_loss improved from 0.34204 to 0.30432, saving model to nn_big_v0_1.h5
21485/21485 [==============================] - 45s 2ms/step - loss: 0.3366 - accuracy: 0.8616 - val_loss: 0.3043 - val_accuracy: 0.8721 - lr: 0.0010
Epoch 3/10
21462/21485 [============================>.] - ETA: 0s - loss: 0.3204 - accuracy: 0.8682
Epoch 3: val_loss improved from 0.30432 to 0.28955, saving model to nn_big_v0_1.h5
21485/21485 [==============================] - 45s 2ms/step - loss: 0.3204 - accuracy: 0.8682 - val_loss: 0.2896 - val_accuracy: 0.8819 - lr: 0.0010
Epoch 4/10
21481/21485 [============================>.] - ETA: 0s - loss: 0.3103 - accuracy: 0.8728
Epoch 4: val_loss did not improve from 0.28955
21485/21485 [==============================] - 45s 2ms/step - loss: 0.3103 - accuracy: 0.8728 - val_loss: 0.2899 - val_accuracy: 0.8831 - lr: 0.0010
Epoch 5/10
21485/21485 [===========================